# Anomaly detection on faulty synthetic test runs

This parallels TEP notebook 2.0. It scores only the endogenous process variables in held-out faulty runs, using healthy-trained autoregressive models and their residual precision matrix. The lag, window, consecutive-alarm count, and threshold are loaded from notebooks 1.0 and 1.1; no setting is refit on test runs. Anomaly labels and fault onset steps come from the synthetic run manifest.

In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [2]:
import numpy as np
import pandas as pd

from carl.causalai_process import ENDOGENOUS_NAMES


data_dir = PROJECT_ROOT / "data" / "interim" / "causalAIProcess"
source_dir = PROJECT_ROOT / "data" / "processed" / "causalAIProcess"
models_dir = PROJECT_ROOT / "models" / "causalAIProcess" / "max_lag_selection"
results_dir = PROJECT_ROOT / "reports" / "results" / "causalAIProcess"
results_dir.mkdir(parents=True, exist_ok=True)

test_file_name = "traces_scaled.parquet"
alarm_results_path = results_dir / "window_and_consecutive_alarm_selection.csv"
thresholds_path = results_dir / "alarm_thresholds_by_window.csv"
selected_lag_path = models_dir / "selected_max_lag.csv"
required_paths = [
    data_dir / test_file_name,
    source_dir / "runs.csv",
    source_dir / "causal_graph.csv",
    alarm_results_path,
    thresholds_path,
    selected_lag_path,
    models_dir / "sample_precision_train.npy",
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Run notebooks 1.0 and 1.1 first. Missing: "
        + ", ".join(str(path) for path in missing_paths)
    )

df = pd.read_parquet(data_dir / test_file_name)
runs = pd.read_csv(source_dir / "runs.csv")
causal_graph = pd.read_csv(source_dir / "causal_graph.csv")
max_lag = int(pd.read_csv(selected_lag_path).loc[0, "best_max_lag"])
alarm_results = pd.read_csv(alarm_results_path)
thresholds_by_window = pd.read_csv(thresholds_path).set_index("Window Size")
f1_col = "Trajectory-Phase F1"
delay_col = "Median Anomaly Detection Delay"
best_f1 = alarm_results[f1_col].max()
best_candidates = alarm_results.loc[alarm_results[f1_col].eq(best_f1)].copy()
best_candidates = best_candidates.dropna(subset=[delay_col])
if best_candidates.empty:
    raise ValueError("No best-F1 configuration has a valid detection delay.")
best_idx = best_candidates[delay_col].idxmin()
best_settings = alarm_results.loc[best_idx]
window = int(best_settings["Window Size"])
consecutive_alarms = int(best_settings["Consecutive Alarms"])
threshold = float(thresholds_by_window.loc[window, "Threshold"])
if not np.isclose(threshold, best_settings["Threshold"]):
    raise ValueError("Selected validation threshold does not match the saved threshold table.")
sample_precision = np.load(models_dir / "sample_precision_train.npy")

feature_names = [
    column
    for column in df.columns
    if column not in ["run_id", "time_step", "time_seconds", "anomaly", "split"]
]
child_parents_pairs = {
    child: sorted(causal_graph.loc[causal_graph["target"] == child, "source"].unique())
    for child in feature_names
}
model_features = {
    child: list(dict.fromkeys(parents + [child]))
    for child, parents in child_parents_pairs.items()
}
process_children = [name for name in feature_names if name in ENDOGENOUS_NAMES]
if set(process_children) != set(ENDOGENOUS_NAMES):
    raise ValueError("Traces are missing endogenous process variables.")

test_run_manifest = runs.loc[
    runs["split"].eq("test") & runs["fault_type"].ne("healthy")
].set_index("run_id")
test_run_ids = test_run_manifest.index.tolist()
df_faulty = df.loc[df["run_id"].isin(test_run_ids)]
if not test_run_ids:
    raise ValueError("No faulty test runs are available for evaluation.")
if df.groupby("run_id").size().nunique() != 1:
    raise ValueError("All trajectories must have equal length for evaluation.")
n_samples_per_faulty_ts = int(df.groupby("run_id").size().iloc[0])
n_faulty_ts = len(test_run_ids)

print(f"Held-out faulty runs: {n_faulty_ts}")
print(f"Selected lag={max_lag}, window={window}, consecutive alarms={consecutive_alarms}")
print(f"Validation-selected threshold={threshold:.6g}")
display(test_run_manifest.groupby("fault_type").size())

Held-out faulty runs: 225
Selected lag=2, window=8, consecutive alarms=5
Validation-selected threshold=2.97599


fault_type
hydraulic_coupling_shift    75
reaction_coupling_shift     75
thermal_coupling_shift      75
dtype: int64

In [3]:
def create_lagged_data(dataframe, child, lag):
    features = dataframe[model_features[child]]
    lagged = pd.concat(
        [features.shift(step).add_suffix(f"_lag{step}") for step in range(1, lag + 1)],
        axis=1,
    ).iloc[lag:]
    x = np.ones((len(lagged), lagged.shape[1] + 1))
    x[:, 1:] = lagged.to_numpy()
    y = dataframe[child].iloc[lag:].to_numpy()
    return x, y


scores_by_run = {}
alarms_by_run = {}
labels_by_run = {}
steps_by_run = {}
score_rows = []

for run_id, trajectory in df_faulty.groupby("run_id", sort=False):
    trajectory = trajectory.sort_values("time_step")
    residuals = np.empty((n_samples_per_faulty_ts - max_lag, len(process_children)))

    for child_idx, child in enumerate(process_children):
        beta_path = models_dir / f"beta_child_{child}_lag_{max_lag}.npy"
        if not beta_path.is_file():
            raise FileNotFoundError(f"Missing fitted autoregressive model: {beta_path}")
        beta = np.load(beta_path)
        x_test, y_test = create_lagged_data(trajectory, child, max_lag)
        residuals[:, child_idx] = y_test - x_test @ beta

    scores = np.mean((residuals @ sample_precision) * residuals, axis=1)
    smoothed_scores = pd.Series(scores).rolling(window=window, min_periods=window).mean()
    raw_breaches = smoothed_scores.gt(threshold).fillna(False).astype(int)
    alarms = (
        raw_breaches.rolling(
            window=consecutive_alarms,
            min_periods=consecutive_alarms,
        )
        .sum()
        .eq(consecutive_alarms)
        .fillna(False)
        .to_numpy(dtype=int)
    )
    steps = trajectory["time_step"].to_numpy()[max_lag:]
    labels = trajectory["anomaly"].to_numpy()[max_lag:]
    scores_by_run[run_id] = scores
    alarms_by_run[run_id] = alarms
    labels_by_run[run_id] = labels
    steps_by_run[run_id] = steps
    score_rows.append(
        pd.DataFrame(
            {
                "run_id": run_id,
                "time_step": steps,
                "anomaly": labels,
                "score": scores,
                "smoothed_score": smoothed_scores.to_numpy(),
                "alarm": alarms,
            }
        )
    )

df_scores = pd.concat(score_rows, ignore_index=True)
df_scores = df_scores.merge(
    test_run_manifest[["fault_type", "fault_start_step"]],
    left_on="run_id",
    right_index=True,
    validate="many_to_one",
)
df_scores.to_parquet(results_dir / "faulty_test_scores.parquet", index=False)
print(f"Scored {len(df_scores):,} held-out faulty samples.")

Scored 224,550 held-out faulty samples.


In [4]:
def binary_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    tpr = tp / (tp + fn) if tp + fn else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * precision * tpr / (precision + tpr) if precision + tpr else 0.0
    return accuracy, tpr, fpr, f1


evaluation_mask = np.ones(n_samples_per_faulty_ts - max_lag, dtype=bool)
evaluation_mask[: window + consecutive_alarms - 2] = False
y_true = np.concatenate([labels_by_run[run_id][evaluation_mask] for run_id in test_run_ids])
y_pred = np.concatenate([alarms_by_run[run_id][evaluation_mask] for run_id in test_run_ids])
sample_accuracy, sample_tpr, sample_fpr, sample_f1 = binary_metrics(y_true, y_pred)

phase_true = []
phase_pred = []
detection_delays = []
per_fault_results = []
for fault_type, fault_manifest in test_run_manifest.groupby("fault_type"):
    fault_true = []
    fault_pred = []
    fault_delays = []
    for run_id, run in fault_manifest.iterrows():
        steps = steps_by_run[run_id]
        alarms = alarms_by_run[run_id]
        before_fault = steps < run["fault_start_step"]
        during_fault = steps >= run["fault_start_step"]
        phase_true.extend([0, 1])
        phase_pred.extend([int(alarms[before_fault].any()), int(alarms[during_fault].any())])
        fault_true.extend([0, 1])
        fault_pred.extend([int(alarms[before_fault].any()), int(alarms[during_fault].any())])
        detected = np.flatnonzero(during_fault & (alarms == 1))
        if len(detected):
            delay = int(steps[detected[0]] - run["fault_start_step"])
            detection_delays.append(delay)
            fault_delays.append(delay)

    fault_accuracy, fault_tpr, fault_fpr, fault_f1 = binary_metrics(
        np.asarray(fault_true), np.asarray(fault_pred)
    )
    per_fault_results.append(
        {
            "Fault Type": fault_type,
            "Runs": len(fault_manifest),
            "Trajectory-Phase Accuracy": fault_accuracy,
            "Trajectory-Phase TPR": fault_tpr,
            "Trajectory-Phase FPR": fault_fpr,
            "Trajectory-Phase F1": fault_f1,
            "Detected Runs": len(fault_delays),
            "Run Detection Rate": len(fault_delays) / len(fault_manifest),
            "Median Detection Delay (seconds)": (
                float(np.median(fault_delays)) if fault_delays else np.nan
            ),
        }
    )

phase_accuracy, phase_tpr, phase_fpr, phase_f1 = binary_metrics(
    np.asarray(phase_true), np.asarray(phase_pred)
)
overall_results = pd.DataFrame(
    [
        {
            "Max Lag": max_lag,
            "Window Size": window,
            "Consecutive Alarms": consecutive_alarms,
            "Threshold": threshold,
            "Sample Accuracy": sample_accuracy,
            "Sample TPR": sample_tpr,
            "Sample FPR": sample_fpr,
            "Sample F1": sample_f1,
            "Trajectory-Phase Accuracy": phase_accuracy,
            "Trajectory-Phase TPR": phase_tpr,
            "Trajectory-Phase FPR": phase_fpr,
            "Trajectory-Phase F1": phase_f1,
            "Detected Runs": len(detection_delays),
            "Total Faulty Runs": n_faulty_ts,
            "Median Detection Delay (seconds)": (
                float(np.median(detection_delays)) if detection_delays else np.nan
            ),
        }
    ]
)
overall_results.to_csv(results_dir / "faulty_test_metrics.csv", index=False)
per_fault_results = pd.DataFrame(per_fault_results)
per_fault_results.to_csv(results_dir / "faulty_test_metrics_by_type.csv", index=False)

display(overall_results)
display(per_fault_results)

,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Detection Delay (seconds)
0,2,8,5,2.97599,0.531226,0.42165,0.0,0.593184,0.917778,0.835556,0.0,0.910412,188,225,31.5


,Fault Type,Runs,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Run Detection Rate,Median Detection Delay (seconds)
0,hydraulic_coupling_shift,75,0.773333,0.546667,0.0,0.706897,41,0.546667,194.0
1,reaction_coupling_shift,75,1.000000,1.000000,0.0,1.000000,75,1.000000,19.0
2,thermal_coupling_shift,75,0.980000,0.960000,0.0,0.979592,72,0.960000,36.0
